# Relax Challenge Take Home Assignment 

The data is available as two attached CSV files:
takehome_user_engagement. csv
takehome_users . csv
The data has the following two tables:
1. A user table ( "takehome_users" ) with data on 12,000 users who signed up for the
product in the last two years. This table includes:
 - name: the user's name
 - object_id: the user's id
 - email: email address
 - creation_source: how their account was created. This takes on one
of 5 values:
   - PERSONAL_PROJECTS: invited to join another user's
personal workspace
   - GUEST_INVITE: invited to an organization as a guest
(limited permissions)
    - ORG_INVITE: invited to an organization (as a full member)
    - SIGNUP: signed up via the website
    - SIGNUP_GOOGLE_AUTH: signed up using Google
Authentication (using a Google email account for their login
id)
- creation_time: when they created their account
- last_session_creation_time: unix timestamp of last login
- opted_in_to_mailing_list: whether they have opted into receiving
marketing emails
- enabled_for_marketing_drip: whether they are on the regular
marketing email drip
- org_id: the organization (group of users) they belong to
- invited_by_user_id: which user invited them to join (if applicable).

2. A usage summary table ( "takehome_user_engagement" ) that has a row for each day
that a user logged into the product.

Defining an "adopted user" as a user who has logged into the product on three separate
days in at least one sevenday
period , identify which factors predict future user
adoption .
We suggest spending 12
hours on this, but you're welcome to spend more or less.
Please send us a brief writeup of your findings (the more concise, the better no
more
than one page), along with any summary tables, graphs, code, or queries that can help
us understand your approach. Please note any factors you considered or investigation
you did, even if they did not pan out. Feel free to identify any further research or data
you think would be valuable.

## Data Wrangling and cleaning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
%matplotlib inline

In [2]:
# Read in the users data to a DataFrame called users
users = pd.read_csv("./takehome_users.csv", encoding="latin-1")

In [3]:
# Review the first 5 rows of the dataframe
users.head()

,object_id,creation_time,name,email,creation_source,last_session_creation_time,opted_in_to_mailing_list,enabled_for_marketing_drip,org_id,invited_by_user_id
0,1,2014-04-22 03:53:30,Clausen August,AugustCClausen@yahoo.com,GUEST_INVITE,1.398139e+09,1,0,11,10803.0
1,2,2013-11-15 03:45:04,Poole Matthew,MatthewPoole@gustr.com,ORG_INVITE,1.396238e+09,0,0,1,316.0
2,3,2013-03-19 23:14:52,Bottrill Mitchell,MitchellBottrill@gustr.com,ORG_INVITE,1.363735e+09,0,0,94,1525.0
3,4,2013-05-21 08:09:28,Clausen Nicklas,NicklasSClausen@yahoo.com,GUEST_INVITE,1.369210e+09,0,0,1,5151.0
4,5,2013-01-17 10:14:20,Raw Grace,GraceRaw@yahoo.com,GUEST_INVITE,1.358850e+09,0,0,193,5240.0


In [7]:
# Check the data info
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   object_id                   12000 non-null  int64  
 1   creation_time               12000 non-null  object 
 2   name                        12000 non-null  object 
 3   email                       12000 non-null  object 
 4   creation_source             12000 non-null  object 
 5   last_session_creation_time  8823 non-null   float64
 6   opted_in_to_mailing_list    12000 non-null  int64  
 7   enabled_for_marketing_drip  12000 non-null  int64  
 8   org_id                      12000 non-null  int64  
 9   invited_by_user_id          6417 non-null   float64
dtypes: float64(2), int64(4), object(4)
memory usage: 937.6+ KB


In [9]:
# Read in the users engagement data to a DataFrame called users
engage = pd.read_csv("./takehome_user_engagement.csv")

In [11]:
# # Review the first 5 rows of the dataframe
engage.head()

,time_stamp,user_id,visited
0,2014-04-22 03:53:30,1,1
1,2013-11-15 03:45:04,2,1
2,2013-11-29 03:45:04,2,1
3,2013-12-09 03:45:04,2,1
4,2013-12-25 03:45:04,2,1


In [13]:
# Check data info
engage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207917 entries, 0 to 207916
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   time_stamp  207917 non-null  object
 1   user_id     207917 non-null  int64 
 2   visited     207917 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.8+ MB


In [15]:
# Check dimensions for both dataframes 
print("Users shape:", users.shape)
print("Engagement shape:", engage.shape)

Users shape: (12000, 10)
Engagement shape: (207917, 3)


In [17]:
# Check for missing info 
print(users.isnull().sum())

object_id                        0
creation_time                    0
name                             0
email                            0
creation_source                  0
last_session_creation_time    3177
opted_in_to_mailing_list         0
enabled_for_marketing_drip       0
org_id                           0
invited_by_user_id            5583
dtype: int64


In [19]:
# Check for missing info 
print(engage.isnull().sum())

time_stamp    0
user_id       0
visited       0
dtype: int64


In [23]:
# Check value counts for visited 
engage['visited'].value_counts()

visited
1    207917
Name: count, dtype: int64

In [25]:
# Change all date variables to datetime
users['creation_time'] = pd.to_datetime(users['creation_time'])

engage['time_stamp'] = pd.to_datetime(engage['time_stamp'])

In [27]:
# Check for unique users in the engagement data 
print("Unique users in engagement data:", engage['user_id'].nunique())

print("Total users:", users['object_id'].nunique())

Unique users in engagement data: 8823
Total users: 12000


In [29]:
# Create the adopted-user target variable 
engage['date'] = engage['time_stamp'].dt.normalize()

In [31]:
# Sort the values 
engage = engage.sort_values(['user_id', 'date'])

In [33]:
# Identify whether each user had 3 visits within any 7-day period
adoption = (
    engage.groupby('user_id')['date']
    .apply(lambda x: ((x.shift(-2) - x).dt.days <= 7).any())
    .reset_index(name='adopted_user')
)

# Convert True/False to 1/0
adoption['adopted_user'] = adoption['adopted_user'].astype(int)

adoption.head()

,user_id,adopted_user
0,1,0
1,2,1
2,3,0
3,4,0
4,5,0


In [35]:
# Add back the users with no engagement
users = users.merge(
    adoption,
    how='left',
    left_on='object_id',
    right_on='user_id'
)

In [37]:
users['adopted_user'] = users['adopted_user'].fillna(0).astype(int)

In [39]:
users.drop(columns='user_id', inplace=True)

In [41]:
users['adopted_user'].value_counts()

adopted_user
0    10344
1     1656
Name: count, dtype: int64

In [43]:
users['adopted_user'].value_counts(normalize=True)

adopted_user
0    0.862
1    0.138
Name: proportion, dtype: float64

##### Observations: 
Based on the definition of adopted users, we find that only 13.8% of our sample are adopted users. 

## Exploratory Analysis

In [45]:
# Adoption rate by source 
adoption_by_source = (
    users.groupby('creation_source')['adopted_user']
    .agg(['count', 'sum', 'mean'])
    .sort_values('mean', ascending=False)
)

adoption_by_source

,count,sum,mean
creation_source,,,
SIGNUP_GOOGLE_AUTH,1385,239,0.172563
GUEST_INVITE,2163,369,0.170596
SIGNUP,2087,302,0.144705
ORG_INVITE,4254,574,0.134932
PERSONAL_PROJECTS,2111,172,0.081478


In [47]:
# Adoption by mailing list
adoption_by_mailing = (
    users.groupby('opted_in_to_mailing_list')['adopted_user']
    .agg(['count', 'sum', 'mean'])
)

adoption_by_mailing

,count,sum,mean
opted_in_to_mailing_list,,,
0,9006,1227,0.136243
1,2994,429,0.143287


In [49]:
# Adoption by marketing drip
adoption_by_drip = (
    users.groupby('enabled_for_marketing_drip')['adopted_user']
    .agg(['count', 'sum', 'mean'])
)

adoption_by_drip

,count,sum,mean
enabled_for_marketing_drip,,,
0,10208,1399,0.137049
1,1792,257,0.143415


In [51]:
# Create invitation status 
users['was_invited'] = users['invited_by_user_id'].notna().astype(int)

In [53]:
# Adoption by invitation status
adoption_by_invitation = (
    users.groupby('was_invited')['adopted_user']
    .agg(['count', 'sum', 'mean'])
)

adoption_by_invitation

,count,sum,mean
was_invited,,,
0,5583,713,0.127709
1,6417,943,0.146953


##### Observations: 

1. Creation source: Adoption rates vary noticeably by account creation source. SIGNUP_GOOGLE_AUTH (17.3%) and GUEST_INVITE (17.1%) have the highest adoption rates, while PERSONAL_PROJECTS has the lowest rate (8.1%), suggesting that creation source may be an important predictor of adoption.
2. Mailing list opt-in: Adoption rates are very similar for users who opted into the mailing list (14.3%) and those who did not (13.6%), suggesting little association with adoption.
3. Marketing drip: Users enabled for the marketing drip have a slightly higher adoption rate (14.3%) than those who are not (13.7%), but the difference is minimal.
4. Invitation status: Users who were invited have a somewhat higher adoption rate (14.7%) than users who were not invited (12.8%), indicating a modest potential association with adoption.

In [55]:
# Check whether adoption differs substantially across organizations.
org_adoption = (
    users.groupby('org_id')['adopted_user']
    .agg(['count', 'sum', 'mean'])
    .sort_values('mean', ascending=False)
)

org_adoption.head(10)

,count,sum,mean
org_id,,,
387,12,7,0.583333
235,13,6,0.461538
270,14,6,0.428571
399,13,5,0.384615
400,8,3,0.375000
392,16,6,0.375000
415,16,6,0.375000
117,22,8,0.363636
345,14,5,0.357143


In [57]:
org_adoption.tail(10)

,count,sum,mean
org_id,,,
242,24,0,0.0
279,17,0,0.0
299,18,0,0.0
307,19,0,0.0
310,15,0,0.0
329,13,0,0.0
346,12,0,0.0
355,9,0,0.0
365,11,0,0.0


In [59]:
print("Number of organizations:", users['org_id'].nunique())

Number of organizations: 417


In [61]:
# Check distribution of org sizes 
users['org_id'].value_counts().describe()

count    417.000000
mean      28.776978
std       27.560173
min        2.000000
25%       17.000000
50%       22.000000
75%       29.000000
max      319.000000
Name: count, dtype: float64

##### Observations 
1. The highest observed adoption rate is about 58.3% for organization 387.
2. Several organizations have adoption rates of 0%.
3. There are 417 organizations in total.
4. Organization sizes vary considerably, ranging from 2 to 319 users.
5. The median organization contains 22 users, while the mean is about 29 users, suggesting that a few relatively large organizations pull the average upward.

In [63]:
# Create a feature for the number of users in each organization
users['org_size'] = users.groupby('org_id')['object_id'].transform('count')

users[['org_id', 'org_size']].head()

,org_id,org_size
0,11,75
1,1,233
2,94,32
3,1,233
4,193,16


In [65]:
users.groupby('adopted_user')['org_size'].describe()

,count,mean,std,min,25%,50%,75%,max
adopted_user,,,,,,,,
0,10344.0,57.258604,66.301696,2.0,21.0,30.0,57.0,319.0
1,1656.0,41.678140,45.133180,7.0,19.0,27.0,42.0,319.0


In [67]:
users[['org_size', 'adopted_user']].corr()

,org_size,adopted_user
org_size,1.000000,-0.083936
adopted_user,-0.083936,1.000000


##### Observations 
Organization size: Organization sizes range from 2 to 319 users. Adopted users belong to somewhat smaller organizations on average (mean = 41.7 users) than non-adopted users (mean = 57.3 users). However, the correlation between organization size and adoption is weak (-0.084), suggesting that organization size alone is unlikely to be a strong predictor.

In [69]:
# How long a user has had an opportunity to become adopted
# Find the end of the observation period
observation_end = engage['time_stamp'].max()

print("Observation period ends:", observation_end)

Observation period ends: 2014-06-06 14:58:50


In [71]:
# Calculate how many days each account existed during the observation period
users['account_age_days'] = (
    observation_end - users['creation_time']
).dt.days

users['account_age_days'].describe()

count    12000.000000
mean       324.568000
std        216.646173
min          6.000000
25%        129.000000
50%        304.000000
75%        506.000000
max        736.000000
Name: account_age_days, dtype: float64

In [73]:
users.groupby('adopted_user')['account_age_days'].describe()

,count,mean,std,min,25%,50%,75%,max
adopted_user,,,,,,,,
0,10344.0,317.094838,217.889945,6.0,117.0,296.0,500.0,736.0
1,1656.0,371.248188,202.624912,18.0,197.0,361.5,548.0,735.0


##### Observations 
Account age: Adopted users had accounts for longer on average during the observation period (mean = 371 days; median = 362 days) than non-adopted users (mean = 317 days; median = 296 days). This suggests that the amount of time a user has been observed may be associated with adoption and should be considered in the predictive analysis.

In [75]:
# Count how many users each inviter invited
inviter_counts = (
    users['invited_by_user_id']
    .value_counts()
    .rename('inviter_invite_count')
)

# Map the count back to each user
users['inviter_invite_count'] = (
    users['invited_by_user_id'].map(inviter_counts)
)

In [77]:
users['inviter_invite_count'] = (
    users['inviter_invite_count']
    .fillna(0)
)

In [79]:
users.groupby('adopted_user')['inviter_invite_count'].describe()

,count,mean,std,min,25%,50%,75%,max
adopted_user,,,,,,,,
0,10344.0,2.094354,2.643721,0.0,0.0,1.0,4.0,13.0
1,1656.0,2.204710,2.580382,0.0,0.0,1.0,4.0,13.0


In [81]:
users[['inviter_invite_count', 'adopted_user']].corr()

,inviter_invite_count,adopted_user
inviter_invite_count,1.000000,0.014444
adopted_user,0.014444,1.000000


##### Observations 
Inviter activity: The number of users invited by a user's inviter showed virtually no relationship with adoption. Adopted and non-adopted users had very similar average inviter invitation counts, and the correlation with adoption was close to zero (0.014). Therefore, this feature is unlikely to be a useful predictor of adoption.

In [83]:
# Check the distribution of two continuous features 
users[['org_size', 'account_age_days']].describe()

,org_size,account_age_days
count,12000.000000,12000.000000
mean,55.108500,324.568000
std,64.023959,216.646173
min,2.000000,6.000000
25%,21.000000,129.000000
50%,29.000000,304.000000
75%,55.000000,506.000000
max,319.000000,736.000000


## Modeling 

In [85]:
model_df = users[
    [
        'creation_source',
        'opted_in_to_mailing_list',
        'enabled_for_marketing_drip',
        'was_invited',
        'org_size',
        'account_age_days',
        'adopted_user'
    ]
].copy()

model_df.head()

,creation_source,opted_in_to_mailing_list,enabled_for_marketing_drip,was_invited,org_size,account_age_days,adopted_user
0,GUEST_INVITE,1,0,1,75,45,0
1,ORG_INVITE,0,0,1,233,203,1
2,ORG_INVITE,0,0,1,32,443,0
3,GUEST_INVITE,0,0,1,233,381,0
4,GUEST_INVITE,0,0,1,16,505,0


In [87]:
model_df.isnull().sum()

creation_source               0
opted_in_to_mailing_list      0
enabled_for_marketing_drip    0
was_invited                   0
org_size                      0
account_age_days              0
adopted_user                  0
dtype: int64

In [89]:
model_df['adopted_user'].value_counts(normalize=True)

adopted_user
0    0.862
1    0.138
Name: proportion, dtype: float64

### Model 1: Logistic Regression

In [91]:
# Set features and target variable 
X = model_df.drop('adopted_user', axis=1)
y = model_df['adopted_user']

In [93]:
# Create train/test split 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [95]:
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining adoption rate:")
print(y_train.value_counts(normalize=True))

print("\nTest adoption rate:")
print(y_test.value_counts(normalize=True))

Training set shape: (9600, 6)
Test set shape: (2400, 6)

Training adoption rate:
adopted_user
0    0.861979
1    0.138021
Name: proportion, dtype: float64

Test adoption rate:
adopted_user
0    0.862083
1    0.137917
Name: proportion, dtype: float64


In [97]:
# Preprocess the features 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [99]:
# Define the column groups 
categorical_features = ['creation_source']

numeric_features = [
    'org_size',
    'account_age_days'
]

binary_features = [
    'opted_in_to_mailing_list',
    'enabled_for_marketing_drip',
    'was_invited'
]

In [101]:
# Create a pre-processor 
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         categorical_features),
        
        ('num', StandardScaler(), numeric_features),
        
        ('bin', 'passthrough', binary_features)
    ]
)

In [103]:
# Create a pipeline 
log_reg = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]
)

In [105]:
# Fit the model 
log_reg.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['creation_source']),
                                                 ('num', StandardScaler(),
                                                  ['org_size',
                                                   'account_age_days']),
                                                 ('bin', 'passthrough',
                                                  ['opted_in_to_mailing_list',
                                                   'enabled_for_marketing_drip',
                                                   'was_invited'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [107]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [109]:
# Generate predicted classes
y_pred = log_reg.predict(X_test)

# Generate predicted probabilities
y_prob = log_reg.predict_proba(X_test)[:, 1]

In [111]:
# Print evaluation metrics 
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

Accuracy: 0.8621
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC-AUC: 0.6094


C:\Users\Abhi\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [113]:
pd.Series(y_prob).describe()

count    2400.000000
mean        0.138840
std         0.054813
min         0.013389
25%         0.102271
50%         0.134146
75%         0.176094
max         0.298114
dtype: float64

In [117]:
print("Maximum predicted probability:", y_prob.max())
print("Minimum predicted probability:", y_prob.min())

Maximum predicted probability: 0.2981141231838432
Minimum predicted probability: 0.013389193556603302


In [119]:
pd.Series(y_pred).value_counts()

0    2400
Name: count, dtype: int64

In [127]:
# Compare model performance across several classification thresholds

thresholds = [0.10, 0.15, 0.20, 0.25]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)
    
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_test, y_pred_threshold),
        'Precision': precision_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        'Recall': recall_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        'F1 Score': f1_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        'Predicted Adopted': y_pred_threshold.sum()
    })

threshold_results = pd.DataFrame(results)

threshold_results

,Threshold,Accuracy,Precision,Recall,F1 Score,Predicted Adopted
0,0.10,0.340000,0.157088,0.867069,0.265987,1827
1,0.15,0.614167,0.180451,0.507553,0.266244,931
2,0.20,0.776250,0.205714,0.217523,0.211454,350
3,0.25,0.844167,0.234568,0.057402,0.092233,81


In [129]:
# Extract regression coefficients 
import numpy as np

# Get feature names after preprocessing
feature_names = log_reg.named_steps['preprocessor'].get_feature_names_out()

# Get logistic regression coefficients
coefficients = log_reg.named_steps['model'].coef_[0]

# Create a table of coefficients and odds ratios
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Odds Ratio': np.exp(coefficients)
})

# Sort by coefficient
coef_df = coef_df.sort_values(
    'Coefficient',
    ascending=False
)

coef_df

,Feature,Coefficient,Odds Ratio
5,num__account_age_days,0.271863,1.312407
6,bin__opted_in_to_mailing_list,0.057193,1.058861
7,bin__enabled_for_marketing_drip,-0.053218,0.948174
3,cat__creation_source_SIGNUP_GOOGLE_AUTH,-0.063612,0.938369
8,bin__was_invited,-0.105028,0.900299
0,cat__creation_source_ORG_INVITE,-0.243035,0.784244
4,num__org_size,-0.332511,0.717120
2,cat__creation_source_SIGNUP,-0.360391,0.697404
1,cat__creation_source_PERSONAL_PROJECTS,-0.972586,0.378104


# Relax Take-Home Challenge: Findings

## Approach

I defined an adopted user as someone who logged into the product on at least three separate days within any rolling seven-day period. This resulted in 1,656 adopted users out of 12,000 total users, for an overall adoption rate of 13.8%.

I explored several potential predictors of adoption, including account creation source, marketing email participation, invitation status, organization size, account age within the observation period, and the number of users invited by the user's inviter. I then fit a logistic regression model using creation source, mailing list participation, marketing drip status, invitation status, organization size, and account age as predictors.

## Key Findings

1. Account creation source showed meaningful differences in adoption rates. Users who signed up using Google authentication and users who joined through a guest invitation had the highest observed adoption rates, at approximately 17%. In contrast, users who joined through personal projects had the lowest adoption rate, at approximately 8%.

2. The logistic regression results reinforced this finding. Using `GUEST_INVITE` as the reference category, `PERSONAL_PROJECTS` was associated with substantially lower odds of adoption (odds ratio = 0.38). `SIGNUP` and `ORG_INVITE` were also associated with lower odds of adoption relative to `GUEST_INVITE`, although the differences were smaller.

3. Account age was positively associated with adoption: a one-standard-deviation increase in account age was associated with approximately 31% higher odds of adoption. This likely reflects, at least in part, the greater opportunity longer-observed users had to meet the adoption definition.

4. Organization size showed a negative association with adoption. A one-standard-deviation increase in organization size was associated with approximately 28% lower odds of adoption. However, the relationship observed during exploratory analysis was relatively weak, so this finding should be interpreted cautiously.

5. Marketing-related variables showed little association with adoption. Mailing list opt-in and participation in the marketing drip both had odds ratios close to 1. Similarly, the number of users invited by a user's inviter showed virtually no relationship with adoption and was excluded from the final model.

## Model Performance and Limitations

The logistic regression achieved a ROC-AUC of 0.61, indicating that the available features contain some predictive signal but have limited ability to distinguish adopted from non-adopted users. At the default classification threshold of 0.50, the model predicted no users as adopted because predicted probabilities did not exceed approximately 0.30. Lowering the threshold improved recall but substantially increased false positives, indicating a clear precision-recall tradeoff.

These results suggest that user characteristics available at signup provide only modest predictive power. Additional behavioral information—such as early product usage, features used, number of sessions, actions completed, or engagement during the first several days after signup—would likely improve the ability to predict future adoption.
